# Learning Rate Schedules

## Historical Context

Learning rate scheduling has evolved significantly:

- **2012-2015**: Step decay and exponential decay were standard
- **2016**: SGDR (Stochastic Gradient Descent with Warm Restarts) introduced cosine annealing with restarts (Loshchilov & Hutter)
- **2017**: Transformer paper used linear warmup + inverse sqrt decay
- **2018**: Super-convergence with 1cycle policy (Smith & Topin)
- **2019**: BERT popularized linear warmup + linear decay
- **2020+**: GPT-3 and modern LLMs typically use cosine decay with warmup

## Why Scheduling Matters

**Warmup rationale**:
- Early training: weights are random, gradients are noisy
- High LR + random weights = unstable training
- Warmup allows model to find a reasonable basin before aggressive optimization
- Critical for transformers with Adam optimizer

**Decay rationale**:
- Early training: need large steps to find good region
- Late training: need small steps to fine-tune
- Annealing helps convergence to sharper minima (better generalization)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable
import math

## 1. Learning Rate Schedule Implementations

In [ ]:
class LRScheduler:
    """Base class for learning rate schedulers."""
    
    def __init__(self, optimizer, max_lr: float):
        self.optimizer = optimizer
        self.max_lr = max_lr
        self.step_count = 0
        
    def step(self):
        """Update learning rate."""
        self.step_count += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr
    
    def get_lr(self) -> float:
        """Calculate learning rate for current step."""
        raise NotImplementedError


class LinearWarmupCosineDecay(LRScheduler):
    """Linear warmup + cosine annealing (most common for LLMs)."""
    
    def __init__(self, optimizer, max_lr: float, warmup_steps: int, 
                 total_steps: int, min_lr: float = 0.0):
        super().__init__(optimizer, max_lr)
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        
    def get_lr(self) -> float:
        # Linear warmup phase
        if self.step_count < self.warmup_steps:
            return self.max_lr * self.step_count / self.warmup_steps
        
        # Cosine decay phase
        progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress = min(progress, 1.0)  # Clamp to [0, 1]
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        return self.min_lr + (self.max_lr - self.min_lr) * cosine_decay


class CosineAnnealingWarmRestarts(LRScheduler):
    """Cosine annealing with warm restarts (SGDR).
    
    Restarts allow model to escape local minima and explore more.
    """
    
    def __init__(self, optimizer, max_lr: float, T_0: int, T_mult: int = 1, 
                 min_lr: float = 0.0):
        super().__init__(optimizer, max_lr)
        self.T_0 = T_0  # Period of first restart
        self.T_mult = T_mult  # Factor to increase period after each restart
        self.min_lr = min_lr
        self.T_cur = 0  # Steps since last restart
        self.T_i = T_0  # Current restart period
        
    def get_lr(self) -> float:
        # Cosine annealing within current period
        cosine_decay = 0.5 * (1 + math.cos(math.pi * self.T_cur / self.T_i))
        lr = self.min_lr + (self.max_lr - self.min_lr) * cosine_decay
        
        # Check if we need to restart
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0
            self.T_i *= self.T_mult
        
        return lr


class LinearWarmupLinearDecay(LRScheduler):
    """Linear warmup + linear decay (used in BERT)."""
    
    def __init__(self, optimizer, max_lr: float, warmup_steps: int, 
                 total_steps: int, min_lr: float = 0.0):
        super().__init__(optimizer, max_lr)
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        
    def get_lr(self) -> float:
        # Linear warmup
        if self.step_count < self.warmup_steps:
            return self.max_lr * self.step_count / self.warmup_steps
        
        # Linear decay
        progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress = min(progress, 1.0)
        return self.max_lr - (self.max_lr - self.min_lr) * progress


class ConstantWithWarmup(LRScheduler):
    """Linear warmup + constant LR (simple but effective for some tasks)."""
    
    def __init__(self, optimizer, max_lr: float, warmup_steps: int):
        super().__init__(optimizer, max_lr)
        self.warmup_steps = warmup_steps
        
    def get_lr(self) -> float:
        if self.step_count < self.warmup_steps:
            return self.max_lr * self.step_count / self.warmup_steps
        return self.max_lr


class PolynomialDecay(LRScheduler):
    """Polynomial decay with warmup.
    
    power=1.0 gives linear decay, power>1 gives slower initial decay.
    """
    
    def __init__(self, optimizer, max_lr: float, warmup_steps: int,
                 total_steps: int, power: float = 1.0, min_lr: float = 0.0):
        super().__init__(optimizer, max_lr)
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.power = power
        self.min_lr = min_lr
        
    def get_lr(self) -> float:
        # Linear warmup
        if self.step_count < self.warmup_steps:
            return self.max_lr * self.step_count / self.warmup_steps
        
        # Polynomial decay
        progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress = min(progress, 1.0)
        decay = (1 - progress) ** self.power
        return self.min_lr + (self.max_lr - self.min_lr) * decay


class InverseSqrtDecay(LRScheduler):
    """Inverse square root decay (original Transformer paper)."""
    
    def __init__(self, optimizer, max_lr: float, warmup_steps: int):
        super().__init__(optimizer, max_lr)
        self.warmup_steps = warmup_steps
        # Scale factor to match max_lr at end of warmup
        self.scale = max_lr * math.sqrt(warmup_steps)
        
    def get_lr(self) -> float:
        # Warmup phase
        if self.step_count < self.warmup_steps:
            return self.max_lr * self.step_count / self.warmup_steps
        
        # Inverse sqrt decay
        return self.scale / math.sqrt(self.step_count)

## 2. Visualize All Schedules

In [ ]:
def visualize_schedules(total_steps: int = 10000, warmup_steps: int = 1000, max_lr: float = 1e-3):
    """Compare all learning rate schedules."""
    
    # Create dummy optimizer
    model = nn.Linear(10, 10)
    
    schedules = {
        'Cosine (LLM standard)': LinearWarmupCosineDecay(
            torch.optim.Adam(model.parameters()), max_lr, warmup_steps, total_steps, min_lr=max_lr*0.1
        ),
        'Cosine w/ Restarts': CosineAnnealingWarmRestarts(
            torch.optim.Adam(model.parameters()), max_lr, T_0=2000, T_mult=2, min_lr=max_lr*0.1
        ),
        'Linear (BERT)': LinearWarmupLinearDecay(
            torch.optim.Adam(model.parameters()), max_lr, warmup_steps, total_steps, min_lr=0
        ),
        'Constant': ConstantWithWarmup(
            torch.optim.Adam(model.parameters()), max_lr, warmup_steps
        ),
        'Polynomial (p=2)': PolynomialDecay(
            torch.optim.Adam(model.parameters()), max_lr, warmup_steps, total_steps, power=2.0
        ),
        'Inverse Sqrt (Transformer)': InverseSqrtDecay(
            torch.optim.Adam(model.parameters()), max_lr, warmup_steps
        ),
    }
    
    # Collect learning rates
    steps = range(1, total_steps + 1)
    lrs = {name: [] for name in schedules.keys()}
    
    for step in steps:
        for name, scheduler in schedules.items():
            lr = scheduler.get_lr()
            lrs[name].append(lr)
            scheduler.step_count += 1
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, (name, lr_values) in enumerate(lrs.items()):
        ax = axes[idx]
        ax.plot(steps, lr_values, linewidth=2)
        ax.set_xlabel('Training Step', fontsize=12)
        ax.set_ylabel('Learning Rate', fontsize=12)
        ax.set_title(name, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
        ax.legend()
    
    plt.tight_layout()
    plt.savefig('lr_schedules_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Plot all together
    plt.figure(figsize=(14, 8))
    for name, lr_values in lrs.items():
        plt.plot(steps, lr_values, linewidth=2, label=name, alpha=0.8)
    
    plt.axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
    plt.xlabel('Training Step', fontsize=14)
    plt.ylabel('Learning Rate', fontsize=14)
    plt.title('Learning Rate Schedule Comparison', fontsize=16, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.savefig('lr_schedules_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_schedules()

## 3. When to Use Which Schedule

### Cosine Annealing (Most Popular)
**Use for**: Pre-training LLMs, fine-tuning large models
- Smooth decay curve
- Proven to work well across scales
- Used by GPT-3, LLaMA, Mistral

### Cosine with Restarts
**Use for**: Training from scratch with limited compute, exploring loss landscape
- Allows model to escape local minima
- Can provide multiple "checkpoints" at cycle ends
- More exploration vs exploitation

### Linear Decay (BERT-style)
**Use for**: Fine-tuning on downstream tasks, classification
- Simple and predictable
- Works well for shorter training runs
- Popular in NLP fine-tuning

### Constant with Warmup
**Use for**: Small-scale experiments, when you want simplicity
- Minimal hyperparameters
- Can work well for LoRA or other parameter-efficient methods

### Polynomial Decay
**Use for**: When you want control over decay curve
- power=1: linear
- power>1: slower initial decay, faster final decay
- Less common in modern LLMs

### Inverse Sqrt
**Use for**: Original Transformer-style training
- Never reaches zero
- Good for continuous/streaming training
- Less common now, but theoretically motivated

## 4. Practical Training Example

In [ ]:
class ToyTransformer(nn.Module):
    """Minimal transformer for demonstration."""
    
    def __init__(self, vocab_size: int = 1000, d_model: int = 128, n_heads: int = 4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=512, batch_first=True)
        self.output = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.output(x)


def train_with_schedule(schedule_name: str, num_steps: int = 2000):
    """Train toy model with specified schedule."""
    
    # Setup
    model = ToyTransformer()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)  # Initial LR will be overridden
    
    # Choose schedule
    warmup_steps = 200
    if schedule_name == 'cosine':
        scheduler = LinearWarmupCosineDecay(optimizer, 1e-3, warmup_steps, num_steps, min_lr=1e-5)
    elif schedule_name == 'linear':
        scheduler = LinearWarmupLinearDecay(optimizer, 1e-3, warmup_steps, num_steps, min_lr=0)
    elif schedule_name == 'constant':
        scheduler = ConstantWithWarmup(optimizer, 1e-3, warmup_steps)
    else:
        raise ValueError(f"Unknown schedule: {schedule_name}")
    
    # Training loop
    losses = []
    lrs = []
    
    model.train()
    for step in range(num_steps):
        # Dummy batch
        batch = torch.randint(0, 1000, (32, 64))
        targets = torch.randint(0, 1000, (32, 64))
        
        # Forward
        logits = model(batch)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, 1000), 
            targets.reshape(-1)
        )
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update LR
        lr = scheduler.step()
        
        losses.append(loss.item())
        lrs.append(lr)
        
        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.4f} | LR: {lr:.6f}")
    
    return losses, lrs


# Compare schedules
results = {}
for schedule in ['cosine', 'linear', 'constant']:
    print(f"\n{'='*60}")
    print(f"Training with {schedule.upper()} schedule")
    print('='*60)
    losses, lrs = train_with_schedule(schedule)
    results[schedule] = (losses, lrs)

In [ ]:
# Visualize training dynamics
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

for name, (losses, lrs) in results.items():
    # Smooth losses with moving average
    window = 50
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    
    ax1.plot(smoothed, label=f'{name.capitalize()}', linewidth=2)
    ax2.plot(lrs, label=f'{name.capitalize()}', linewidth=2)

ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Loss (smoothed)', fontsize=12)
ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Training Step', fontsize=12)
ax2.set_ylabel('Learning Rate', fontsize=12)
ax2.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('schedule_training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Benchmarks and Best Practices

In [ ]:
def analyze_schedule_impact(total_steps: int = 5000):
    """Analyze how warmup ratio affects training."""
    
    model = nn.Linear(10, 10)
    max_lr = 1e-3
    
    warmup_ratios = [0.01, 0.05, 0.1, 0.2, 0.3]
    
    plt.figure(figsize=(14, 8))
    
    for ratio in warmup_ratios:
        warmup_steps = int(total_steps * ratio)
        scheduler = LinearWarmupCosineDecay(
            torch.optim.Adam(model.parameters()), 
            max_lr, warmup_steps, total_steps, min_lr=max_lr*0.1
        )
        
        lrs = []
        for step in range(1, total_steps + 1):
            scheduler.step_count = step
            lrs.append(scheduler.get_lr())
        
        plt.plot(lrs, label=f'Warmup: {ratio*100:.0f}% ({warmup_steps} steps)', linewidth=2)
    
    plt.xlabel('Training Step', fontsize=14)
    plt.ylabel('Learning Rate', fontsize=14)
    plt.title('Impact of Warmup Ratio', fontsize=16, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.savefig('warmup_ratio_impact.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Recommendations
    print("Warmup Ratio Recommendations:")
    print("  - Short training (<1000 steps): 5-10% warmup")
    print("  - Medium training (1000-10000 steps): 10% warmup")
    print("  - Long training (>10000 steps): 1-5% warmup")
    print("  - Very long (100k+ steps): <1% warmup (but min 100-1000 steps)")

analyze_schedule_impact()

## 6. Best Practices Summary

### Warmup Duration
- **Too short**: Training instability, loss spikes
- **Too long**: Wasted training time
- **Rule of thumb**: 5-10% of total steps, minimum 100-1000 steps

### Maximum Learning Rate
- **Adam/AdamW**: Start with 1e-4 to 1e-3
- **SGD**: Start with 1e-1 to 1e-2 (much higher)
- **LoRA/PEFT**: Can use higher LR (1e-3 to 1e-4)
- **Full fine-tuning**: Lower LR (1e-5 to 1e-4)

### Minimum Learning Rate
- **Non-zero minimum** (0.1x max_lr): Prevents complete stagnation
- **Zero minimum**: Fine for fixed-length training

### Schedule Choice
1. **Default**: Cosine with warmup (safest, most popular)
2. **Fine-tuning**: Linear or cosine (both work)
3. **Experimentation**: Try restarts for exploration
4. **Simple baseline**: Constant with warmup

### Modern LLM Practices (2024-2025)
- GPT-3/4: Cosine decay with warmup
- LLaMA: Cosine decay, 2000 warmup steps
- Mistral: Cosine decay with warmup
- Gemma: Cosine decay

**Common pattern**: `warmup_steps=min(2000, 0.05 * total_steps)`

## 7. Integration with Popular Frameworks

In [ ]:
# Using transformers library
from transformers import get_scheduler

def transformers_scheduler_example():
    """Show how to use transformers library schedulers."""
    
    model = nn.Linear(10, 10)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    
    # Available types: linear, cosine, cosine_with_restarts, polynomial, constant, constant_with_warmup
    scheduler = get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=500,
        num_training_steps=5000
    )
    
    print("Transformers Scheduler Configuration:")
    print(f"  Type: cosine")
    print(f"  Warmup steps: 500")
    print(f"  Total steps: 5000")
    print(f"  Max LR: {optimizer.param_groups[0]['lr']}")
    
    # In training loop:
    # optimizer.step()
    # scheduler.step()
    
    return scheduler

transformers_scheduler_example()

In [ ]:
# PyTorch native schedulers
def pytorch_scheduler_example():
    """Show PyTorch native scheduler options."""
    
    model = nn.Linear(10, 10)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    # Option 1: CosineAnnealingLR
    scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=1000, eta_min=1e-5
    )
    
    # Option 2: OneCycleLR (super-convergence)
    scheduler2 = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3, total_steps=1000
    )
    
    # Option 3: ReduceLROnPlateau (adaptive)
    scheduler3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    print("PyTorch Native Schedulers:")
    print("  1. CosineAnnealingLR - Smooth cosine decay")
    print("  2. OneCycleLR - Super-convergence with single cycle")
    print("  3. ReduceLROnPlateau - Adaptive based on metrics")
    print("\nNote: For transformers, custom warmup schedulers (like above) are preferred")

pytorch_scheduler_example()

## Key Takeaways

1. **Warmup is critical** for transformer training stability
2. **Cosine annealing** is the modern default for LLMs
3. **Warmup ratio**: 5-10% of total steps (min 100-1000 steps)
4. **Schedule choice** matters less than getting max_lr and warmup right
5. **Modern practice**: Linear warmup + cosine decay to 0.1x max_lr
6. **Fine-tuning**: Can use simpler schedules (linear, constant)
7. **Experimentation**: Try different schedules if training is unstable